# Create anndata for peakVI

In [1]:
here::i_am("atac/archR/dimensionality_reduction/peakVI/peakVI_get_anndata.ipynb")

suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(scater))


# Load default settings
source(here::here("settings.R"))
source(here::here("utils.R"))

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/09_Eomes_invitro_blood/code



In [2]:
args <- list()
args$sce <-file.path(io$basedir,"processed/atac/archR/Matrices/PeakMatrix_summarized_experiment.rds")
args$metadata <- file.path(io$basedir,"results/atac/archR/qc/sample_metadata_after_qc.txt.gz")
args$outdir = file.path(io$basedir, 'results/atac/archR/dimensionality_reduction/peakVI/')
# I/O
dir.create(args$outdir, showWarnings=F, recursive=T)

In [8]:
##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadata) %>%
  .[pass_rnaQC==TRUE & doublet_call==FALSE & pass_atacQC==TRUE] %>%
  .[,exp:=str_replace_all(sample, opts$sample2exp)]

In [7]:
# get Variable features
source(here::here("atac/archR/load_archR_project.R"))

# Subset
ArchRProject.filt <- ArchRProject[sample_metadata$cell]

# Iterative LSI: two iterations
ArchRProject.filt <- addIterativeLSI(
  ArchRProj = ArchRProject.filt,
  useMatrix = 'PeakMatrix', 
  name = "IterativeLSI", 
  firstSelection = "Top",
  depthCol = "nFrags",
  iterations = 2, 
   clusterParams = list(
    resolution = 0.3, 
    sampleCells = 10000, 
    maxClusters=NULL,
    n.start = 10
   ), 
  saveIterations = FALSE,
  varFeatures = 35000, 
  dimsToUse = 1:15,
  force = TRUE
)

LSI <- getReducedDims(ArchRProject.filt, "IterativeLSI", returnMatrix=F)


Setting default genome to Mm10.

Setting default number of Parallel threads to 1.

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\__

In [9]:
features = as.data.table(LSI$LSIFeatures)[,feature:=paste0(seqnames, ':', start, '-', end)]$feature

In [15]:
write.table(features, sprintf("%s/features.txt.gz",args$outdir))

In [4]:
features = read.table(sprintf("%s/features.txt.gz",args$outdir))$x

In [5]:
sce = readRDS(args$sce)
sce = sce[,sample_metadata$cell]
sce = as(sce, 'SingleCellExperiment')

sce = sce[features,]

assay(sce, 'counts') = assay(sce, 'PeakMatrix')
assay(sce, 'PeakMatrix') = NULL

colData(sce) <- sample_metadata %>% as.data.frame %>% tibble::column_to_rownames("cell") %>%
  .[colnames(sce),] %>% DataFrame()

In [10]:
# Anndata with only top 45k variable features
sceasy::convertFormat(sce, from="sce", 
                      to="anndata",
                      outFile= sprintf("%s/anndata_ATAC_variable.h5ad",args$outdir))

Warning message in .regularise_df(as.data.frame(SummarizedExperiment::colData(obj)), :
“Dropping single category variables:pass_rnaQC, doublet_call, pass_atacQC”


AnnData object with n_obs × n_vars = 33177 × 35000
    obs: 'barcode', 'sample', 'nFeature_RNA', 'nCount_RNA', 'mitochondrial_percent_RNA', 'ribosomal_percent_RNA', 'alias', 'day', 'genotype', 'doublet_score', 'celltype', 'celltype.score', 'closest.cell', 'day_celltype', 'celltype_genotype', 'TSSEnrichment_atac', 'ReadsInTSS_atac', 'PromoterRatio_atac', 'NucleosomeRatio_atac', 'nFrags_atac', 'BlacklistRatio_atac', 'exp'
    var: 'idx'